# Bundled Exasol lesson: First Steps

Uses the kit encrypted connection. Database examples use EXAKIT_EXAMPLES. Later sections may need tutorial downloads or the ML UDF add-on.

Run cells from top to bottom. In Marimo each editable cell has its own Run button; opening a lesson executes no example code. Widget steps retain their Python code; use the matching Jupyter notebook for the original widget UI.

For dashboards use the kit Data Scientist 03_share flow on Dash Server.

Source: [Exasol Notebook Connector](https://github.com/exasol/notebook-connector/blob/14e2c7ad088b7109aa21d283e98f3126bbbe27e5/exasol/nb_connector/resources/notebooks/first_steps.ipynb). MIT license: LICENSE.txt.

In [ ]:
from exakit.nb.example_runtime import bootstrap
ai_lab_config = bootstrap(globals(), 'first_steps.ipynb')

# First Steps

This notebook guides you through the first steps using Jupyter notebooks with Exasol.

The notebook demonstrates connecting to an Exasol database instance and using some of its features. 

## 1. Open Secure Configuration Storage

First we need to open the Secure Configuration Storage (SCS) containing the connection information such as the database host, user, password, etc.

In [ ]:
print("Using the Starter Kit encrypted connection; tutorial schema: EXAKIT_EXAMPLES.")

## 2. Using JupySQL with Exasol AI Lab

Next, we will activate the [JupySQL](https://github.com/ploomber/jupysql) magics:

In [ ]:
from exakit.nb.example_runtime import init_sql as jupysql_init
jupysql_init(ai_lab_config)

JupySQL allows you to run SQL and plot large datasets in Jupyter via a %sql, %%sql, and %sqlplot magics. Using these magics, you can enter SQL directly into notebook cells.
In the background JupySQL uses SQLAlchemy, see also the [demo section on SQLAlchemy](./first_steps.ipynb/#4.-Using-SQLAlchemy-with-Exasol-AI-Lab) below.

### 2.1 Create Database Tables

We will use JupySQL to create 2 database tables but other sections will use the tables, too:

In [ ]:
%%sql
CREATE OR REPLACE TABLE US_AIRLINES (
  OP_CARRIER_AIRLINE_ID DECIMAL(10, 0) IDENTITY PRIMARY KEY,
  CARRIER_NAME VARCHAR(1000)
)

In [ ]:
%%sql
CREATE OR REPLACE TABLE US_FLIGHTS (
  FL_DATE TIMESTAMP, 
  OP_CARRIER_AIRLINE_ID DECIMAL(10, 0),
  ORIGIN_AIRPORT_SEQ_ID DECIMAL(10, 0),
  ORIGIN_STATE_ABR CHAR(2),
  DEST_AIRPORT_SEQ_ID DECIMAL(10, 0),
  DEST_STATE_ABR CHAR(2),
  CRS_DEP_TIME CHAR(4),
  DEP_DELAY DOUBLE, 
  CRS_ARR_TIME CHAR(4),
  ARR_DELAY DOUBLE,
  CANCELLED BOOLEAN,
  CANCELLATION_CODE CHAR(1),
  DIVERTED BOOLEAN,
  CRS_ELAPSED_TIME DOUBLE,
  ACTUAL_ELAPSED_TIME DOUBLE, 
  DISTANCE DOUBLE,
  CARRIER_DELAY DOUBLE,
  WEATHER_DELAY DOUBLE,
  NAS_DELAY DOUBLE,
  SECURITY_DELAY DOUBLE,
  LATE_AIRCRAFT_DELAY DOUBLE
)

### 2.2 Importing CSV Files from Remote

This section demonstrates how to import CSV files from a remote source into the database.

First we will import a list of **US airlines**. The data is publicly accessible at the [Bureau of Transportation Statistics](https://www.transtats.bts.gov/Homepage.asp) of the US Department of Transportation.

In [ ]:
%%sql
IMPORT INTO US_AIRLINES FROM
    CSV AT 'https://dut5tonqye28.cloudfront.net/ai_lab/flight-info/' 
    FILE 'US_AIRLINES.csv' 
    COLUMN SEPARATOR = ',' 
    ROW SEPARATOR = 'CRLF'
    COLUMN DELIMITER = '"' 
    SKIP = 1

Next, we will import data about **flights** in February 2024:

In [ ]:
%%sql
IMPORT INTO US_FLIGHTS 
    FROM CSV AT 'https://dut5tonqye28.cloudfront.net/ai_lab/first_steps/' 
    FILE 'US_FLIGHTS_FEB_2024-fixed-booleans.csv'
    (1 FORMAT = 'MM/DD/YYYY HH12:MI:SS AM', 2..21)    
    SKIP = 1

Let's find out which is the airline with the highest delay per flight:

In [ ]:
%%sql
SELECT
  CARRIER_NAME "Airline",
  SUM(CARRIER_DELAY) "Combined Delay",
  COUNT(CARRIER_DELAY) "Delayed Flights",
  COUNT(F.OP_CARRIER_AIRLINE_ID) "Total flights",
  ROUND( SUM(CARRIER_DELAY) / COUNT(F.OP_CARRIER_AIRLINE_ID), 1 ) "Delay per flight"
FROM US_FLIGHTS F
  JOIN US_AIRLINES A ON A.OP_CARRIER_AIRLINE_ID = F.OP_CARRIER_AIRLINE_ID
WHERE NOT (CANCELLED OR DIVERTED)
GROUP BY CARRIER_NAME
ORDER BY "Delay per flight" DESC

### 2.3 Importing a Parquet File from an AWS S3 Bucket

This demo uses a file already uploaded to S3 bucket `ai-lab-example-data-s3`.

First we will define a connection pointing to the S3 bucket:

In [ ]:
%%sql
CREATE OR REPLACE CONNECTION AI_LAB_FIRST_STEPS_S3 
    TO 'https://ai-lab-example-data-s3.s3.eu-central-1.amazonaws.com'

Alternatively the connection can also use the following URL syntax, see also "_Load data from Parquet files_" on [docs.exasol.com](https://docs.exasol.com/db/latest/loading_data/load_data_parquet.htm#Overview):

In [ ]:
%%sql
CREATE OR REPLACE CONNECTION AI_LAB_FIRST_STEPS_S3 
    TO 's3://ai-lab-example-data-s3'

Then we will remove the data imported before:

In [ ]:
%%sql
TRUNCATE TABLE US_FLIGHTS

**Please note**: Parquet import via SQL statement `IMPORT` requires using **<span style="color: #40a">Exasol version 2025.\* or higher</span>**, see [docs.exasol.com](https://docs.exasol.com/db/latest/loading_data/load_data_parquet.htm).

Now we can import the Parquet file from S3 into the database:

In [ ]:
%%sql

IMPORT INTO US_FLIGHTS FROM PARQUET AT AI_LAB_FIRST_STEPS_S3 
    FILE 'first_steps/US_FLIGHTS_FEB_2024.parquet'

We will query table `US_FLIGHTS` again to display the imported data:

In [ ]:
%%sql
SELECT * FROM US_FLIGHTS

## 3. Using PyExasol with Exasol AI Lab

[PyExasol](https://exasol.github.io/pyexasol/master/index.html) is the basic Python connector for interacting with Exasol databases.

### 3.1 Importing a CSV File from the Local Filesystem

This section demonstrates how to import a CSV file from the local file system into the database using Pyexasol.

The function `open_pyexasol_connection()` opens a connection, using the configuration from the SCS.

In [ ]:
from pathlib import Path
from exakit.nb.example_runtime import open_pyexasol_connection

with open_pyexasol_connection(ai_lab_config, compression=True) as conn:
    path = Path("first_steps/pyexasol.csv")
    import_params = {
        "column_delimiter": '"',
        "column_separator": ",",
        "row_separator": "CRLF",
        "skip": 1,
    }
    conn.import_from_file(path, (ai_lab_config.db_schema, "US_AIRLINES"), import_params)

We will also verify the success by using a query with separate parameters and [PyExasol SQL formatting](https://exasol.github.io/pyexasol/master/user_guide/exploring_features/formatting_sql.html):

In [ ]:
from exakit.nb.example_runtime import open_pyexasol_connection

query = "SELECT * FROM {table!q} WHERE CARRIER_NAME LIKE {pattern!s}"
params = {
    "table": (ai_lab_config.db_schema, "US_AIRLINES"),
    "pattern": "% local CSV file via pyexasol",
}

with open_pyexasol_connection(ai_lab_config, compression=True) as conn:
    result = conn.execute(query, params).fetchone()
print(result)

### 3.2 Importing a CSV File from Remote

This section demonstrates how to import a CSV file from a remote source into the database using PyExasol.

First, we will truncate table `US_FLIGHTS` to be able to import the flight data again:

In [ ]:
%%sql
TRUNCATE TABLE US_FLIGHTS

Now let's run the import&mdash;again using a query with separate parameter and PyExasol SQL formatting:

In [ ]:
from exakit.nb.example_runtime import open_pyexasol_connection

query = """
    IMPORT INTO {flights_table!q} FROM CSV AT {url!s} FILE {file!s}
    (1 FORMAT = {date_format!s}, 2..21) 
    SKIP = 1
"""

params = {
    "flights_table": (ai_lab_config.db_schema, "US_FLIGHTS"),
    "url": "https://dut5tonqye28.cloudfront.net/ai_lab/first_steps/",
    "file": "US_FLIGHTS_FEB_2024-fixed-booleans.csv",
    "date_format": "MM/DD/YYYY HH12:MI:SS AM",
}

with open_pyexasol_connection(ai_lab_config, compression=True) as conn:
    result = conn.execute(query, params)

print(f"Imported {result.rowcount()} rows.")

### 3.3 Importing a Parquet File from an AWS S3 Bucket

This demo uses 
* a file already uploaded to S3 bucket `ai-lab-example-data-s3` 
* and the Exasol connection object `AI_LAB_FIRST_STEPS_S3` created before.

First we will truncate table `US_FLIGHTS`:

In [ ]:
%%sql
TRUNCATE TABLE US_FLIGHTS

**Please note**: 
* Parquet import via SQL statement `IMPORT` requires using **<span style="color: #40a">Exasol version 2025.\* or higher</span>**, see [docs.exasol.com](https://docs.exasol.com/db/latest/loading_data/load_data_parquet.htm).
* This restriction does not apply when using PyExasol's native function [import_from_parquet()](https://exasol.github.io/pyexasol/master/user_guide/exploring_features/import_and_export/index.html#id8).

Now we can import the Parquet file from S3 into the database:

In [ ]:
from exakit.nb.example_runtime import open_pyexasol_connection

query = "IMPORT INTO {table!q} FROM PARQUET AT {connection!r} FILE {file!s}"

params = {
    "table": (ai_lab_config.db_schema, "US_FLIGHTS"),
    "connection": "AI_LAB_FIRST_STEPS_S3",
    "file": "first_steps/US_FLIGHTS_FEB_2024.parquet",
}

with open_pyexasol_connection(ai_lab_config, compression=True) as conn:
    result = conn.execute(query, params)
print(f"Imported {result.rowcount()} rows.")

## 4. Using SQLAlchemy with Exasol AI Lab

[SQLAlchemy](https://www.sqlalchemy.org/) is a Python SQL Toolkit and Object Relational Mapper for application developers.
                                                               
It provides a full suite of well known enterprise-level persistence patterns, designed for efficient and high-performing database access, adapted into a simple and Pythonic domain language.

In Exasol AI Lab, the function `open_sqlalchemy_connection()` returns a SQLAlchemy engine, again using the configuration from the SCS.

This engine will be used by the examples based on SQLAlchemy.

In [ ]:
from exakit.nb.example_runtime import open_sqlalchemy_connection
engine = open_sqlalchemy_connection(ai_lab_config)

### 4.1 Importing a CSV File from Remote

This section demonstrates how to import a CSV file from a remote source into the database using SQLAlchemy.

We will once again truncate table `US_FLIGHTS` and import the flight data, now using SQLAlchemy:

In [ ]:
from sqlalchemy import text

t = (
    text(
        f"""
        IMPORT INTO "{ai_lab_config.db_schema}"."US_FLIGHTS" 
        FROM CSV AT 'https://dut5tonqye28.cloudfront.net/ai_lab/first_steps/' 
        FILE 'US_FLIGHTS_FEB_2024-fixed-booleans.csv'
        (1 FORMAT = 'MM/DD/YYYY HH12:MI:SS AM', 2..21) 
        SKIP = 1
        """
    )
)
with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE US_FLIGHTS"))
    result = conn.execute(t)

print(f"Imported {result.rowcount} rows.")

### 4.2 Importing a Parquet File from an AWS S3 Bucket

This section demonstrates how to import a Parquet file from an AWS S3 Bucket into the database using SQLAlchemy.

First we will truncate table `US_FLIGHTS` again:

In [ ]:
%%sql
TRUNCATE TABLE US_FLIGHTS

**Please note**: Parquet import via SQL statement `IMPORT` requires using **<span style="color: #40a">Exasol version 2025.\* or higher</span>**, see [docs.exasol.com](https://docs.exasol.com/db/latest/loading_data/load_data_parquet.htm).

Now we can import the Parquet file from S3 into the database:

In [ ]:
from sqlalchemy import text

t = (
    text(
        f"""
        IMPORT INTO "{ai_lab_config.db_schema}"."US_FLIGHTS" 
        FROM PARQUET AT AI_LAB_FIRST_STEPS_S3
        FILE 'first_steps/US_FLIGHTS_FEB_2024.parquet'
        """
    )
)

with engine.connect() as conn:
    result = conn.execute(t)
print(f"Imported {result.rowcount} rows.")

### 4.3 Querying using SQLAlchemy’s Metadata Concept

This section demonstrates using some SQLAlchemy features 
* Instantiating class [sqlalchemy.MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData)
* Querying existing SQL tables with [.reflect()](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData)
* Retrieving a specific table 
* Using its [select](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table.select) statement and
* Executing it in a SQLAlchemy [Session](https://docs.sqlalchemy.org/en/20/orm/session_api.html#sqlalchemy.orm.Session) object
* Iterating the result set

In [ ]:
from sqlalchemy import MetaData
from sqlalchemy.orm import Session

meta = MetaData()
db_schema = ai_lab_config.db_schema
meta.reflect(engine, schema=db_schema)

flights = meta.tables[f"{db_schema}.us_flights"]
stmt = (
    flights.select()
    .where(flights.c.op_carrier_airline_id == 20452)
    .where(flights.c.origin_state_abr == "TX")
    .where(flights.c.dest_state_abr == "NJ")
)
with Session(engine) as session:
    for r in session.execute(stmt):
        print(f'Departure: {r[0].date()} {r[6]}')    

## 5. Using the Exasol Bucket File System

The [Exasol Bucket File System](https://docs.exasol.com/db/latest/database_concepts/bucketfs/bucketfs.htm) (BucketFS) is a powerful feature for exchanging non-relational data with the database nodes in an Exasol cluster.

Such data can be arbitrary files including 
* Data to be processed by [User Defined Scripts](https://docs.exasol.com/db/latest/database_concepts/udf_scripts.htm) (UDFs)
* [Script-Languages Containers](https://github.com/exasol/script-languages-release) (SLCs), see also the dedicated [Tutorial on Script-Languages Containers](script_languages_container/using_the_script_languages_container_tool.ipynb)
* AI Models

### 5.1 Uploading a File to the BucketFS

First we will create a sample file:

In [ ]:
%%writefile first_steps/text_file.txt
Hello World!\n

And now, let's upload the file into the BucketFS.

The function `open_bucketfs_location()` returns a path into Exasols BucketFS, also using the configuration in the SCS.

In [ ]:
from exakit.nb.example_runtime import open_bucketfs_location
from pathlib import Path 

file = Path("first_steps/text_file.txt")
bfs = open_bucketfs_location(ai_lab_config)
file_in_bfs = bfs / file.name
file_in_bfs.write(file.read_bytes())

### 5.2 Listing the Files in the BucketFS

We can also list all the files currently available in the BucketFS:

In [ ]:
from exakit.nb.example_runtime import open_bucketfs_location

bfs = open_bucketfs_location(ai_lab_config)
for p in bfs.iterdir():
    print(f'- {p.name}')

### 5.3 Reading the File in the BucketFS Using a User Defined Function

This sections demonstrates reading the contents of a file in the BucketFS using a [User Defined Function](https://docs.exasol.com/db/latest/database_concepts/udf_scripts.htm) (UDF). 

UDFs can be used in Exasol SQL statements and can also access objects in the BucketFS.

First we will define the UDF:

In [ ]:
%%sql
--/
CREATE OR REPLACE PYTHON3_ML SCALAR SCRIPT 
    bfs_file_content("path" VARCHAR(256)) 
    EMITS ("FILE" VARCHAR(256), CONTENT VARCHAR(2000000)) AS
from pathlib import Path 
def run(ctx):  
    ctx.emit(ctx.path, Path(ctx.path).read_text())
/

And now, let's read the contents of the file in the BucketFS we just created before. 

The file's BucketFS location is represented by an instance of class [exasol.bucketfs.PathLike](https://exasol.github.io/bucketfs-python/main/api.html#exasol.bucketfs._path.PathLike) and the method `as_udf_path()` returns the file's path as seen by a UDF:

In [ ]:
%%sql
SELECT bfs_file_content('{{file_in_bfs.as_udf_path()}}');